In [2]:
import kagglehub
import os

path = kagglehub.dataset_download("meowmeowmeowmeowmeow/gtsrb-german-traffic-sign")
print("Path:", path)

c:\Users\gurra\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path: C:\Users\gurra\.cache\kagglehub\datasets\meowmeowmeowmeowmeow\gtsrb-german-traffic-sign\versions\1


In [3]:
train_dir = os.path.join(path, "Train")

In [4]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMG_SIZE = (32, 32)
BATCH_SIZE = 32

datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

train_data = datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='training'
)

val_data = datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='sparse',
    subset='validation'
)

Found 31368 images belonging to 43 classes.
Found 7841 images belonging to 43 classes.


In [5]:
import numpy as np

# Example mapping (adjust if needed)
selected_classes = [14, 17, 2]

def filter_generator(generator):
    while True:
        X, y = next(generator)
        mask = np.isin(y, selected_classes)
        yield X[mask], y[mask]

train_data_filtered = filter_generator(train_data)
val_data_filtered = filter_generator(val_data)

In [6]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([
    Conv2D(32, (3,3), activation='relu', input_shape=(32,32,3)),
    MaxPooling2D(2,2),

    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2,2),

    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),

    Dense(43, activation='softmax')  # full dataset
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

c:\Users\gurra\AppData\Local\Programs\Python\Python313\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 30, 30, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 15, 15, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 13, 13, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 6, 6, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 4, 4, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 2, 2, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │        65,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 43)             │         5,547 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 164,459 (642.42 KB)

 Trainable params: 164,459 (642.42 KB)

 Non-trainable params: 0 (0.00 B)

In [7]:
history = model.fit(
    train_data,
    validation_data=val_data,
    epochs=10
)

Epoch 1/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 64s 63ms/step - accuracy: 0.4860 - loss: 1.7930 - val_accuracy: 0.8235 - val_loss: 0.6362
Epoch 2/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 25s 25ms/step - accuracy: 0.8674 - loss: 0.4162 - val_accuracy: 0.9120 - val_loss: 0.3121
Epoch 3/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 25s 26ms/step - accuracy: 0.9313 - loss: 0.2179 - val_accuracy: 0.9373 - val_loss: 0.2274
Epoch 4/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 25s 25ms/step - accuracy: 0.9561 - loss: 0.1436 - val_accuracy: 0.9374 - val_loss: 0.2192
Epoch 5/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 25s 25ms/step - accuracy: 0.9657 - loss: 0.1142 - val_accuracy: 0.9459 - val_loss: 0.1807
Epoch 6/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 25s 26ms/step - accuracy: 0.9738 - loss: 0.0838 - val_accuracy: 0.9301 - val_loss: 0.2474
Epoch 7/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 25s 26ms/step - accuracy: 0.9787 - loss: 0.0676 - val_accuracy: 0.9489 - val_loss: 0.2320
Epoch 8/10
981/981 ━━━━━━━━━━━━━━━━━━━━ 25s 26ms/step - accuracy: 0.9815 - loss: 0.0586 - 

In [8]:
loss, acc = model.evaluate(val_data)
print("Accuracy:", acc)

246/246 ━━━━━━━━━━━━━━━━━━━━ 4s 17ms/step - accuracy: 0.9591 - loss: 0.2007
Accuracy: 0.959061324596405


In [9]:
import cv2

class_names = list(train_data.class_indices.keys())

def predict_image(img_path):
    img = cv2.imread(img_path)
    img = cv2.resize(img, (32,32))
    img = img / 255.0
    img = np.expand_dims(img, axis=0)

    pred = model.predict(img)
    return class_names[np.argmax(pred)]

In [10]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("traffic_model.tflite", "wb") as f:
    f.write(tflite_model)

INFO:tensorflow:Assets written to: C:\Users\gurra\AppData\Local\Temp\tmpmtsybhuz\assets


INFO:tensorflow:Assets written to: C:\Users\gurra\AppData\Local\Temp\tmpmtsybhuz\assets


Saved artifact at 'C:\Users\gurra\AppData\Local\Temp\tmpmtsybhuz'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 43), dtype=tf.float32, name=None)
Captures:
  2502620741264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2502620740496: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2502620740112: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2502634980752: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2502634980560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2502634981136: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2502634979792: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2502634981328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2502634980368: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2502634981712: TensorSpec(shape=(), dtype=tf.resource, name=None)
